# 🛡️ Network IDS — Streamlit on Google Colab
Run your deep learning intrusion detection app directly in Colab with a public URL via **ngrok**.

### Steps:
1. Get a free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
2. Paste it in **Cell 2**
3. Run all cells in order — a public URL will appear at the end

In [1]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
!pip install -q streamlit pyngrok xgboost scikit-learn plotly pandas numpy joblib
print('✅ All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 80.0 MB/s eta 0:00:00
✅ All packages installed


In [2]:
# ── Cell 2: Set your ngrok token ─────────────────────────────────────────────
# Get yours FREE at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '3DIun2J1uyfIMEFhLqD832c6yBS_6EMce6BQyjwJaU4ovg6Z2'  # <-- replace this

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('✅ ngrok token set')

✅ ngrok token set


In [3]:
# ── Cell 3: Write the Streamlit app to disk ───────────────────────────────────
app_code = '''
"""
🛡️ Real-Time Network IDS — Streamlit Interface
Deep Learning (ANN + LSTM) · Random Forest · XGBoost · Ensemble
"""

import streamlit as st
import pandas as pd
import numpy as np
import random
import time
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="🛡️ Network IDS",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
  [data-testid="stAppViewContainer"] { background: #0d1117; }
  [data-testid="stSidebar"] { background: #161b22; }
  .main-header {
      background: linear-gradient(135deg, #1a1f2e 0%, #0d1117 100%);
      border: 1px solid #30363d; border-radius: 12px;
      padding: 24px; margin-bottom: 24px; text-align: center;
  }
  .normal-badge   { background:#0d3321; color:#00cc88; border:1px solid #00cc88; border-radius:6px; padding:2px 10px; font-weight:700; }
  .suspicious-badge { background:#3d2b00; color:#ffa500; border:1px solid #ffa500; border-radius:6px; padding:2px 10px; font-weight:700; }
  .malicious-badge { background:#3d0d0d; color:#ff4444; border:1px solid #ff4444; border-radius:6px; padding:2px 10px; font-weight:700; }
  .blocked-badge  { background:#4d0000; color:#ff0000; border:1px solid #ff0000; border-radius:6px; padding:2px 10px; font-weight:700; }
  div[data-testid="metric-container"] { background:#161b22; border:1px solid #30363d; border-radius:10px; padding:12px; }
</style>
""", unsafe_allow_html=True)

FEATURE_COLS = [
    "duration", "protocol_type", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised",
    "count", "srv_count", "serror_rate", "rerror_rate",
    "same_srv_rate", "diff_srv_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_serror_rate",
    "dst_host_rerror_rate", "src_port", "dst_port", "packet_size"
]
LABEL_MAP = {0: "Normal", 1: "Suspicious", 2: "Malicious"}
COLOR_MAP  = {"Normal": "#00cc88", "Suspicious": "#ffa500", "Malicious": "#ff4444"}

GEO_DB = {
    "192.168": {"country": "Local LAN",    "city": "Internal",   "lat": 0.0,    "lon": 0.0,    "flag": "🏠"},
    "10.0":    {"country": "Private Net",  "city": "Internal",   "lat": 0.0,    "lon": 0.0,    "flag": "🔒"},
    "8.8":     {"country": "United States","city": "Mountain View","lat": 37.386,"lon": -122.08,"flag": "🇺🇸"},
    "1.1":     {"country": "Australia",    "city": "Sydney",     "lat": -33.86, "lon": 151.20, "flag": "🇦🇺"},
    "185.":    {"country": "Netherlands",  "city": "Amsterdam",  "lat": 52.37,  "lon": 4.90,   "flag": "🇳🇱"},
    "45.":     {"country": "Russia",       "city": "Moscow",     "lat": 55.75,  "lon": 37.62,  "flag": "🇷🇺"},
    "91.":     {"country": "China",        "city": "Beijing",    "lat": 39.90,  "lon": 116.39, "flag": "🇨🇳"},
    "77.":     {"country": "Germany",      "city": "Frankfurt",  "lat": 50.11,  "lon": 8.68,   "flag": "🇩🇪"},
    "203.":    {"country": "India",        "city": "Mumbai",     "lat": 19.08,  "lon": 72.88,  "flag": "🇮🇳"},
    "117.":    {"country": "Japan",        "city": "Tokyo",      "lat": 35.68,  "lon": 139.69, "flag": "🇯🇵"},
    "62.":     {"country": "United Kingdom","city": "London",    "lat": 51.50,  "lon": -0.12,  "flag": "🇬🇧"},
    "200.":    {"country": "Brazil",       "city": "São Paulo",  "lat": -23.54, "lon": -46.63, "flag": "🇧🇷"},
}

EXTERNAL_POOLS = [
    "8.8.{}.{}", "185.{}.{}.{}", "45.{}.{}.{}",
    "91.{}.{}.{}", "77.{}.{}.{}", "203.{}.{}.{}",
    "117.{}.{}.{}", "62.{}.{}.{}", "200.{}.{}.{}",
]

def make_normal(n):
    return pd.DataFrame({
        "duration": np.random.uniform(0,100,n), "protocol_type": np.random.choice([0,1,2],n),
        "src_bytes": np.random.uniform(0,50000,n), "dst_bytes": np.random.uniform(0,50000,n),
        "land": np.zeros(n), "wrong_fragment": np.zeros(n), "urgent": np.zeros(n),
        "hot": np.random.uniform(0,5,n), "num_failed_logins": np.zeros(n),
        "logged_in": np.ones(n), "num_compromised": np.zeros(n),
        "count": np.random.uniform(1,50,n), "srv_count": np.random.uniform(1,50,n),
        "serror_rate": np.random.uniform(0,0.1,n), "rerror_rate": np.random.uniform(0,0.1,n),
        "same_srv_rate": np.random.uniform(0.7,1.0,n), "diff_srv_rate": np.random.uniform(0,0.3,n),
        "dst_host_count": np.random.uniform(100,255,n), "dst_host_srv_count": np.random.uniform(100,255,n),
        "dst_host_same_srv_rate": np.random.uniform(0.7,1.0,n), "dst_host_diff_srv_rate": np.random.uniform(0,0.3,n),
        "dst_host_serror_rate": np.random.uniform(0,0.1,n), "dst_host_rerror_rate": np.random.uniform(0,0.1,n),
        "src_port": np.random.randint(1024,65535,n), "dst_port": np.random.choice([80,443,22,53,8080],n),
        "packet_size": np.random.uniform(64,1500,n), "label": np.zeros(n,dtype=int), "attack_type": ["Normal"]*n,
    })

def make_suspicious(n):
    return pd.DataFrame({
        "duration": np.random.uniform(0,300,n), "protocol_type": np.random.choice([0,1,2],n),
        "src_bytes": np.random.uniform(0,200000,n), "dst_bytes": np.random.uniform(0,10000,n),
        "land": np.random.choice([0,1],n,p=[0.8,0.2]), "wrong_fragment": np.random.uniform(0,3,n),
        "urgent": np.random.choice([0,1],n,p=[0.7,0.3]), "hot": np.random.uniform(5,20,n),
        "num_failed_logins": np.random.uniform(1,5,n), "logged_in": np.random.choice([0,1],n),
        "num_compromised": np.random.uniform(0,5,n), "count": np.random.uniform(100,255,n),
        "srv_count": np.random.uniform(50,200,n), "serror_rate": np.random.uniform(0.3,0.7,n),
        "rerror_rate": np.random.uniform(0.2,0.6,n), "same_srv_rate": np.random.uniform(0.3,0.7,n),
        "diff_srv_rate": np.random.uniform(0.3,0.7,n), "dst_host_count": np.random.uniform(50,255,n),
        "dst_host_srv_count": np.random.uniform(10,100,n), "dst_host_same_srv_rate": np.random.uniform(0.3,0.7,n),
        "dst_host_diff_srv_rate": np.random.uniform(0.3,0.7,n), "dst_host_serror_rate": np.random.uniform(0.3,0.7,n),
        "dst_host_rerror_rate": np.random.uniform(0.2,0.5,n), "src_port": np.random.randint(1,1024,n),
        "dst_port": np.random.choice([22,23,3389,445],n), "packet_size": np.random.uniform(64,3000,n),
        "label": np.ones(n,dtype=int), "attack_type": list(np.random.choice(["Probe","R2L"],n)),
    })

def make_malicious(n):
    return pd.DataFrame({
        "duration": np.random.uniform(0,10,n), "protocol_type": np.random.choice([0,1],n),
        "src_bytes": np.random.uniform(0,1e6,n), "dst_bytes": np.zeros(n),
        "land": np.ones(n), "wrong_fragment": np.random.uniform(3,10,n),
        "urgent": np.ones(n), "hot": np.random.uniform(20,100,n),
        "num_failed_logins": np.random.uniform(5,20,n), "logged_in": np.zeros(n),
        "num_compromised": np.random.uniform(5,50,n), "count": np.full(n,255),
        "srv_count": np.full(n,255), "serror_rate": np.random.uniform(0.8,1.0,n),
        "rerror_rate": np.random.uniform(0.8,1.0,n), "same_srv_rate": np.random.uniform(0,0.2,n),
        "diff_srv_rate": np.random.uniform(0.8,1.0,n), "dst_host_count": np.full(n,255),
        "dst_host_srv_count": np.full(n,255), "dst_host_same_srv_rate": np.random.uniform(0,0.2,n),
        "dst_host_diff_srv_rate": np.random.uniform(0.8,1.0,n), "dst_host_serror_rate": np.random.uniform(0.8,1.0,n),
        "dst_host_rerror_rate": np.random.uniform(0.8,1.0,n), "src_port": np.random.randint(1,100,n),
        "dst_port": np.random.choice([80,443,22,3306,6379],n), "packet_size": np.random.uniform(1500,65535,n),
        "label": np.full(n,2,dtype=int), "attack_type": list(np.random.choice(["DoS","U2R"],n)),
    })

@st.cache_resource(show_spinner=False)
def train_models():
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neural_network import MLPClassifier
    import xgboost as xgb

    df = pd.concat([make_normal(4000), make_suspicious(1500), make_malicious(800)], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    X = df[FEATURE_COLS].fillna(0).values
    y = df["label"].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

    rf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    xgb_model = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08,
        subsample=0.8, colsample_bytree=0.8, eval_metric="mlogloss", random_state=42, n_jobs=-1, verbosity=0)
    xgb_model.fit(X_train, y_train)

    ann = MLPClassifier(hidden_layer_sizes=(256,128,64,32), activation="relu", max_iter=60, early_stopping=True, random_state=42)
    ann.fit(X_train, y_train)

    fi = pd.DataFrame({"feature": FEATURE_COLS, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
    return scaler, rf, xgb_model, ann, fi, X_test, y_test

def geo_lookup(ip):
    for prefix, info in GEO_DB.items():
        if str(ip).startswith(prefix):
            return info
    return {"country": "Unknown", "city": "Unknown", "lat": random.uniform(-60,70), "lon": random.uniform(-150,150), "flag": "🌐"}

def random_external_ip():
    tmpl = random.choice(EXTERNAL_POOLS)
    return tmpl.format(random.randint(1,254), random.randint(1,254), random.randint(1,254))

def compute_risk_score(label, confidence):
    return min(round({0:0,1:40,2:80}.get(label,0) + confidence*15, 1), 100.0)

def classify_single(row_dict, scaler, rf, xgb_model, ann):
    row = {col: row_dict.get(col, 0) for col in FEATURE_COLS}
    X_pkt = scaler.transform(pd.DataFrame([row]).values)
    proba = (rf.predict_proba(X_pkt) + xgb_model.predict_proba(X_pkt) + ann.predict_proba(X_pkt)) / 3
    label = int(np.argmax(proba, axis=1)[0])
    confidence = float(np.max(proba))
    return {"label": LABEL_MAP[label], "label_id": label, "spam": "SPAM" if label>0 else "NOT SPAM",
            "confidence": confidence, "risk_score": compute_risk_score(label, confidence),
            "rf_vote": LABEL_MAP[int(rf.predict(X_pkt)[0])],
            "xgb_vote": LABEL_MAP[int(xgb_model.predict(X_pkt)[0])],
            "ann_vote": LABEL_MAP[int(ann.predict(X_pkt)[0])],
            "proba": proba[0].tolist()}

def classify_batch(df, scaler, rf, xgb_model, ann):
    X_b = scaler.transform(df[FEATURE_COLS].fillna(0).values)
    proba = (rf.predict_proba(X_b) + xgb_model.predict_proba(X_b) + ann.predict_proba(X_b)) / 3
    preds = np.argmax(proba, axis=1)
    confs = np.max(proba, axis=1)
    out = df.copy()
    out["predicted_label_name"] = [LABEL_MAP[p] for p in preds]
    out["spam_flag"]  = ["SPAM" if p>0 else "NOT SPAM" for p in preds]
    out["confidence"] = confs.round(4)
    out["risk_score"] = [compute_risk_score(p,c) for p,c in zip(preds,confs)]
    return out

def generate_live_packet(ext_ratio=0.3):
    roll = random.random()
    if   roll < 0.70: row = make_normal(1);     lt = "normal"
    elif roll < 0.88: row = make_suspicious(1); lt = "suspicious"
    else:             row = make_malicious(1);  lt = "malicious"
    pkt = row.iloc[0].to_dict()
    pkt["timestamp"] = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    pkt["src_ip"] = random_external_ip() if (random.random()<ext_ratio and lt!="normal") else f"192.168.{random.randint(0,255)}.{random.randint(1,254)}"
    pkt["dst_ip"] = f"10.0.{random.randint(0,10)}.{random.randint(1,50)}"
    pkt["src_geo"] = geo_lookup(pkt["src_ip"])
    return pkt

# ── Sidebar ────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🛡️ Network IDS")
    st.markdown("**Deep Learning · Ensemble**")
    st.divider()
    page = st.radio("Navigate", ["📊 Dashboard","🔴 Live Monitor","📁 CSV Analysis","🗺️ Threat Map","🧠 Model Info"])
    st.divider()
    st.markdown("**Settings**")
    block_threshold = st.slider("Auto-Block Threshold", 0.70, 0.99, 0.90, 0.01)
    sim_speed = st.slider("Live Speed (sec)", 0.3, 3.0, 1.0, 0.1)
    ext_ratio = st.slider("External IP Ratio", 0.0, 1.0, 0.3, 0.05)

with st.spinner("⚙️ Training AI models…"):
    scaler, rf, xgb_model, ann, fi, X_test, y_test = train_models()

# ══════════════ DASHBOARD ════════════════════════════════════════════════════
if page == "📊 Dashboard":
    st.markdown("""
    <div class="main-header">
      <h1>🛡️ Network Intrusion Detection System</h1>
      <p style=\'color:#8b949e\'>Deep Learning Ensemble · ANN + XGBoost + Random Forest</p>
    </div>""", unsafe_allow_html=True)

    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
    y_pred_rf  = rf.predict(X_test)
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_ann = ann.predict(X_test)
    p_ens = (rf.predict_proba(X_test) + xgb_model.predict_proba(X_test) + ann.predict_proba(X_test)) / 3
    y_pred_ens = np.argmax(p_ens, axis=1)

    acc = {"RF": accuracy_score(y_test,y_pred_rf), "XGB": accuracy_score(y_test,y_pred_xgb),
           "ANN": accuracy_score(y_test,y_pred_ann), "Ensemble": accuracy_score(y_test,y_pred_ens)}
    f1  = {"RF": f1_score(y_test,y_pred_rf,average="weighted"), "XGB": f1_score(y_test,y_pred_xgb,average="weighted"),
           "ANN": f1_score(y_test,y_pred_ann,average="weighted"), "Ensemble": f1_score(y_test,y_pred_ens,average="weighted")}

    c1,c2,c3,c4 = st.columns(4)
    c1.metric("🎯 Ensemble Acc", f"{acc[\'Ensemble\']:.2%}")
    c2.metric("📈 Ensemble F1",  f"{f1[\'Ensemble\']:.4f}")
    c3.metric("🌲 RF Acc",       f"{acc[\'RF\']:.2%}")
    c4.metric("⚡ XGB Acc",      f"{acc[\'XGB\']:.2%}")
    st.divider()

    col1,col2 = st.columns(2)
    with col1:
        st.subheader("Model Comparison")
        names = list(acc.keys())
        fig = go.Figure()
        fig.add_bar(name="Accuracy", x=names, y=list(acc.values()), marker_color="#00cc88")
        fig.add_bar(name="F1 Score", x=names, y=list(f1.values()),  marker_color="#3b82f6")
        fig.update_layout(barmode="group", paper_bgcolor="#161b22", plot_bgcolor="#0d1117",
                          font_color="#c9d1d9", yaxis=dict(range=[0.8,1.0],gridcolor="#30363d"))
        st.plotly_chart(fig, use_container_width=True)
    with col2:
        st.subheader("Feature Importances")
        top_fi = fi.head(12)
        fig2 = px.bar(top_fi, x="importance", y="feature", orientation="h",
                      color="importance", color_continuous_scale="teal")
        fig2.update_layout(paper_bgcolor="#161b22", plot_bgcolor="#0d1117",
                           font_color="#c9d1d9", yaxis={"categoryorder":"total ascending"}, coloraxis_showscale=False)
        st.plotly_chart(fig2, use_container_width=True)

    st.subheader("Ensemble Confusion Matrix")
    cm = confusion_matrix(y_test, y_pred_ens)
    labels = ["Normal","Suspicious","Malicious"]
    fig3 = px.imshow(cm, text_auto=True, x=labels, y=labels, color_continuous_scale="Blues",
                     labels=dict(x="Predicted", y="Actual"))
    fig3.update_layout(paper_bgcolor="#161b22", font_color="#c9d1d9", coloraxis_showscale=False)
    st.plotly_chart(fig3, use_container_width=True)

# ══════════════ Live Monitor ═════════════════════════════════════════════════

elif page == "🔴 Live Monitor":
    st.markdown("## 🔴 Live Network Traffic Monitor")

    if "live_log" not in st.session_state:
        st.session_state.live_log = []
    if "blocked_ips" not in st.session_state:
        st.session_state.blocked_ips = set()
    if "running" not in st.session_state:
        st.session_state.running = False
    if "counters" not in st.session_state:
        st.session_state.counters = {"normal":0,"suspicious":0,"malicious":0,"total":0}
    if "traffic_series" not in st.session_state:
        st.session_state.traffic_series = []
    if "attack_series" not in st.session_state:
        st.session_state.attack_series = []
    # ── NEW: alert history store ──────────────────────────────────────────
    if "alert_history" not in st.session_state:
        st.session_state.alert_history = []

    c1, c2, c3 = st.columns(3)

    if c1.button("▶ Start", type="primary", use_container_width=True):
        st.session_state.running = True
        st.markdown("""
        <script>
        (function() {
            if (!window._idsAudioCtx) {
                try {
                    window._idsAudioCtx = new (window.AudioContext || window.webkitAudioContext)();
                    window._idsAudioCtx.resume().then(function() {
                        sessionStorage.setItem('idsAudioUnlocked', '1');
                    });
                } catch(e) {}
            }
        })();
        </script>
        """, unsafe_allow_html=True)

    if c2.button("⏹ Stop", use_container_width=True):
        st.session_state.running = False

    if c3.button("🗑 Clear", use_container_width=True):
        st.session_state.live_log = []
        st.session_state.blocked_ips = set()
        st.session_state.counters = {"normal":0,"suspicious":0,"malicious":0,"total":0}
        st.session_state.traffic_series = []
        st.session_state.attack_series = []
        st.session_state.alert_history = []   # clear alert history too

    cnt = st.session_state.counters
    m1,m2,m3,m4,m5 = st.columns(5)
    m1.metric("📦 Total",cnt["total"])
    m2.metric("✅ Normal",cnt["normal"])
    m3.metric("⚠️ Suspicious",cnt["suspicious"])
    m4.metric("🚨 Malicious",cnt["malicious"])
    m5.metric("🔒 Blocked",len(st.session_state.blocked_ips))

    graph_ph = st.empty()
    log_ph = st.empty()
    toast_ph = st.empty()

    # ── TOAST FUNCTION (unchanged) ────────────────────────────────────────
    def show_toast(message, color, sound=False):
        audio = ""
        if sound:
            audio = """
            <script>
            (function() {
                function playBeep() {
                    try {
                        var ctx = window._idsAudioCtx;
                        if (!ctx) {
                            ctx = new (window.AudioContext || window.webkitAudioContext)();
                            window._idsAudioCtx = ctx;
                        }
                        var resumePromise = ctx.state === 'suspended' ? ctx.resume() : Promise.resolve();
                        resumePromise.then(function() {
                            var oscillator = ctx.createOscillator();
                            var gainNode = ctx.createGain();
                            oscillator.connect(gainNode);
                            gainNode.connect(ctx.destination);
                            oscillator.type = 'square';
                            oscillator.frequency.setValueAtTime(880, ctx.currentTime);
                            oscillator.frequency.setValueAtTime(440, ctx.currentTime + 0.1);
                            gainNode.gain.setValueAtTime(0.3, ctx.currentTime);
                            gainNode.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + 0.4);
                            oscillator.start(ctx.currentTime);
                            oscillator.stop(ctx.currentTime + 0.4);
                        });
                    } catch(e) {
                        console.warn('IDS Audio error:', e);
                    }
                }
                if (sessionStorage.getItem('idsAudioUnlocked') === '1' || window._idsAudioCtx) {
                    playBeep();
                }
            })();
            </script>
            """
        toast_ph.markdown(f"""
        <div style="
            position: fixed;
            top: 80px;
            right: 20px;
            background: {color};
            color: white;
            padding: 14px 20px;
            border-radius: 10px;
            box-shadow: 0 0 15px rgba(0,0,0,0.6);
            z-index: 9999;
            font-weight: bold;
            min-width: 260px;
            animation: slidein 0.4s;
        ">
            {message}
        </div>
        <script>
        setTimeout(function(){{
            const el = window.parent.document.querySelectorAll('div[style*="position: fixed"]');
            if(el.length>0){{
                el[el.length-1].remove();
            }}
        }}, 4000);
        </script>
        <style>
        @keyframes slidein {{
            from {{ transform: translateX(100%); opacity:0; }}
            to {{ transform: translateX(0); opacity:1; }}
        }}
        </style>
        {audio}
        """, unsafe_allow_html=True)

    # ── RENDER LOG (unchanged) ────────────────────────────────────────────
    def render_log(log):
        rows = []
        for e in reversed(log[-50:]):
            lbl = e["label"]
            badge = (f"<span class='normal-badge'>{lbl}</span>" if lbl=="Normal"
                else f"<span class='suspicious-badge'>{lbl}</span>" if lbl=="Suspicious"
                else f"<span class='malicious-badge'>{lbl}</span>")
            blk = "<span class='blocked-badge'>🔴 BLOCKED</span>" if e.get("blocked") else ""
            g = e.get("src_geo",{})
            rows.append(f"<tr style='border-bottom:1px solid #21262d'>"
                f"<td style='color:#8b949e;font-size:11px;padding:4px 8px'>{e['timestamp']}</td>"
                f"<td style='padding:4px 8px;color:#c9d1d9'>{e['src_ip']}</td>"
                f"<td style='padding:4px 8px;color:#8b949e'>{g.get('flag','🌐')} {g.get('country','')}</td>"
                f"<td style='padding:4px 8px'>{badge}</td>"
                f"<td style='padding:4px 8px;color:#c9d1d9'>{e['risk_score']}</td>"
                f"<td style='padding:4px 8px;color:#c9d1d9'>{e['confidence']:.1%}</td>"
                f"<td style='padding:4px 8px'>{blk}</td></tr>")
        return ("<div style='background:#0d1117;border:1px solid #30363d;border-radius:10px;overflow:auto;max-height:440px'>"
                "<table style='width:100%;border-collapse:collapse;font-size:12px'>"
                "<thead><tr style='background:#161b22;color:#8b949e'>"
                "<th style='padding:8px'>Time</th><th>Src IP</th><th>Origin</th>"
                "<th>Label</th><th>Risk</th><th>Conf</th><th>Action</th></tr></thead>"
                "<tbody>" + "".join(rows) + "</tbody></table></div>")

    log_ph.markdown(render_log(st.session_state.live_log), unsafe_allow_html=True)

    # ── MAIN LOOP (unchanged logic, alert_history append added) ──────────
    if st.session_state.running:
        pkt = generate_live_packet(ext_ratio)
        result = classify_single(pkt, scaler, rf, xgb_model, ann)

        is_blocked = result["label_id"] == 2 and result["confidence"] >= block_threshold
        if is_blocked:
            st.session_state.blocked_ips.add(pkt["src_ip"])

        entry = {
            "timestamp": pkt["timestamp"],
            "src_ip": pkt["src_ip"],
            "dst_ip": pkt["dst_ip"],
            "label": result["label"],
            "attack_type": pkt.get("attack_type",""),
            "risk_score": result["risk_score"],
            "confidence": result["confidence"],
            "blocked": is_blocked,
            "src_geo": pkt.get("src_geo",{})
        }
        st.session_state.live_log.append(entry)

        # ── NEW: push Suspicious/Malicious to alert history ───────────────
        if result["label"] in ("Suspicious", "Malicious"):
            st.session_state.alert_history.append({
                "timestamp": pkt["timestamp"],
                "src_ip":    pkt["src_ip"],
                "label":     result["label"],
                "risk_score":result["risk_score"],
                "blocked":   is_blocked,
            })
            # keep only last 100 alerts in memory
            if len(st.session_state.alert_history) > 100:
                st.session_state.alert_history.pop(0)

        # toast triggers (unchanged)
        if result["label"] == "Malicious":
            show_toast(
                f"🚨 CRITICAL ATTACK<br>IP: {pkt['src_ip']}<br>Risk: {result['risk_score']}",
                "#ff4444", sound=True
            )
        elif result["label"] == "Suspicious":
            show_toast(
                f"⚠️ Suspicious Activity<br>IP: {pkt['src_ip']}",
                "#ffa500", sound=False
            )

        # graph update (unchanged)
        st.session_state.traffic_series.append(st.session_state.counters["total"])
        st.session_state.attack_series.append(st.session_state.counters["malicious"])
        if len(st.session_state.traffic_series) > 30:
            st.session_state.traffic_series.pop(0)
            st.session_state.attack_series.pop(0)

        lname = result["label"].lower()
        st.session_state.counters["total"] += 1
        st.session_state.counters[lname] = st.session_state.counters.get(lname, 0) + 1

        if len(st.session_state.traffic_series) > 1:
            df_graph = pd.DataFrame({
                "Time": list(range(len(st.session_state.traffic_series))),
                "Total Traffic": st.session_state.traffic_series,
                "Malicious": st.session_state.attack_series
            })
            fig_graph = px.line(
                df_graph, x="Time", y=["Total Traffic","Malicious"],
                title="📊 Live Traffic vs Attacks"
            )
            fig_graph.update_layout(
                paper_bgcolor="#161b22",
                plot_bgcolor="#0d1117",
                font_color="#c9d1d9"
            )
            graph_ph.plotly_chart(fig_graph, use_container_width=True)

        log_ph.markdown(render_log(st.session_state.live_log), unsafe_allow_html=True)
        time.sleep(sim_speed)
        st.rerun()

    if st.session_state.blocked_ips:
        st.error("🔒 Blocked IPs: " + "  |  ".join(list(st.session_state.blocked_ips)[:10]))

    # ═══════════════════════════════════════════════════════════════════════
    # NEW FEATURE 1 — TOP ATTACKING IPs  (table + bar chart, auto-updates)
    # ═══════════════════════════════════════════════════════════════════════
    st.markdown("---")
    st.markdown("### 🔺 Top Attacking IPs")

    # Build attack-count dict from live_log
    attack_ip_counts = {}
    for e in st.session_state.live_log:
        if e["label"] in ("Suspicious", "Malicious"):
            ip = e["src_ip"]
            attack_ip_counts[ip] = attack_ip_counts.get(ip, 0) + 1

    if attack_ip_counts:
        # Sort and take top 10
        top_ips = sorted(attack_ip_counts.items(), key=lambda x: x[1], reverse=True)[:10]
        df_top = pd.DataFrame(top_ips, columns=["IP Address", "Attack Count"])

        col_tbl, col_bar = st.columns([1, 1])

        with col_tbl:
            # Table with SOC colour coding
            rows_html = ""
            for i, row in df_top.iterrows():
                count = row["Attack Count"]
                if count >= 10:
                    color = "#ff4444"
                    badge = f"<span style='background:#ff4444;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px'>{count}</span>"
                elif count >= 5:
                    color = "#ffa500"
                    badge = f"<span style='background:#ffa500;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px'>{count}</span>"
                else:
                    color = "#238636"
                    badge = f"<span style='background:#238636;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px'>{count}</span>"
                rows_html += (
                    f"<tr style='border-bottom:1px solid #21262d'>"
                    f"<td style='padding:5px 10px;color:#c9d1d9;font-size:12px'>#{i+1}</td>"
                    f"<td style='padding:5px 10px;color:#58a6ff;font-size:12px;font-family:monospace'>{row['IP Address']}</td>"
                    f"<td style='padding:5px 10px;text-align:center'>{badge}</td>"
                    f"</tr>"
                )
            st.markdown(
                f"""<div style='background:#0d1117;border:1px solid #30363d;border-radius:10px;overflow:auto'>
                <table style='width:100%;border-collapse:collapse'>
                <thead><tr style='background:#161b22;color:#8b949e;font-size:12px'>
                <th style='padding:8px 10px'>#</th>
                <th style='padding:8px 10px'>IP Address</th>
                <th style='padding:8px 10px;text-align:center'>Hits</th>
                </tr></thead>
                <tbody>{rows_html}</tbody>
                </table></div>""",
                unsafe_allow_html=True
            )

        with col_bar:
            fig_top = px.bar(
                df_top[::-1],          # reverse so highest is at top
                x="Attack Count",
                y="IP Address",
                orientation="h",
                color="Attack Count",
                color_continuous_scale=["#238636", "#ffa500", "#ff4444"],
                title="",
            )
            fig_top.update_layout(
                paper_bgcolor="#0d1117",
                plot_bgcolor="#0d1117",
                font_color="#c9d1d9",
                coloraxis_showscale=False,
                margin=dict(l=10, r=10, t=10, b=10),
                height=300,
                yaxis=dict(tickfont=dict(size=10, family="monospace")),
                xaxis=dict(tickfont=dict(size=10)),
            )
            fig_top.update_traces(marker_line_width=0)
            st.plotly_chart(fig_top, use_container_width=True)
    else:
        st.info("No attack IPs detected yet. Start the monitor to populate this panel.")

    # ═══════════════════════════════════════════════════════════════════════
    # NEW FEATURE 2 — ALERT HISTORY  (last 20 alerts, never missed)
    # ═══════════════════════════════════════════════════════════════════════
    st.markdown("---")
    st.markdown("### 🕐 Alert History")

    if st.session_state.alert_history:
        alert_rows = ""
        for a in reversed(st.session_state.alert_history[-20:]):
            if a["label"] == "Malicious":
                lbl_html = "<span style='background:#ff4444;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px'>🚨 Malicious</span>"
            else:
                lbl_html = "<span style='background:#ffa500;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px'>⚠️ Suspicious</span>"

            blk_html = (
                "<span style='background:#6e40c9;color:#fff;padding:2px 6px;border-radius:4px;font-size:10px'>🔴 BLOCKED</span>"
                if a.get("blocked") else
                "<span style='color:#8b949e;font-size:11px'>—</span>"
            )
            alert_rows += (
                f"<tr style='border-bottom:1px solid #21262d'>"
                f"<td style='color:#8b949e;font-size:11px;padding:5px 10px;white-space:nowrap'>{a['timestamp']}</td>"
                f"<td style='color:#58a6ff;font-size:12px;padding:5px 10px;font-family:monospace'>{a['src_ip']}</td>"
                f"<td style='padding:5px 10px'>{lbl_html}</td>"
                f"<td style='color:#c9d1d9;font-size:12px;padding:5px 10px;text-align:center'>{a['risk_score']}</td>"
                f"<td style='padding:5px 10px;text-align:center'>{blk_html}</td>"
                f"</tr>"
            )
        st.markdown(
            f"""<div style='background:#0d1117;border:1px solid #30363d;border-radius:10px;overflow:auto;max-height:340px'>
            <table style='width:100%;border-collapse:collapse'>
            <thead><tr style='background:#161b22;color:#8b949e;font-size:12px;position:sticky;top:0'>
            <th style='padding:8px 10px'>Time</th>
            <th style='padding:8px 10px'>Src IP</th>
            <th style='padding:8px 10px'>Alert</th>
            <th style='padding:8px 10px;text-align:center'>Risk</th>
            <th style='padding:8px 10px;text-align:center'>Action</th>
            </tr></thead>
            <tbody>{alert_rows}</tbody>
            </table></div>""",
            unsafe_allow_html=True
        )
        st.caption(f"Showing last {min(20, len(st.session_state.alert_history))} of {len(st.session_state.alert_history)} total alerts this session")
    else:
        st.info("No alerts yet. Suspicious and Malicious events will appear here automatically.")

# ══════════════ CSV ANALYSIS ═════════════════════════════════════════════════
elif page == "📁 CSV Analysis":
    st.markdown("## 📁 Batch CSV Analysis")
    with st.expander("📋 Required columns"):
        st.code(", ".join(FEATURE_COLS))

    tab1, tab2, tab3 = st.tabs(["📤 Upload CSV", "🔬 Sample Data", "🧪 Manual Analysis"])

    with tab1:
        uploaded = st.file_uploader("Upload network traffic CSV", type=["csv"])
        if uploaded:
            df_up = pd.read_csv(uploaded)
            st.success(f"Loaded {len(df_up):,} rows")
            st.dataframe(df_up.head(5), use_container_width=True)
            if st.button("🚀 Analyse", type="primary"):
                with st.spinner("Running ensemble AI…"):
                    results = classify_batch(df_up, scaler, rf, xgb_model, ann)
                    if "src_ip" in results.columns:
                        results["src_country"] = results["src_ip"].apply(lambda ip: geo_lookup(str(ip)).get("country",""))
                        results["src_flag"]    = results["src_ip"].apply(lambda ip: geo_lookup(str(ip)).get("flag","🌐"))
                st.session_state["csv_results"] = results
                st.success("✅ Done!")

    with tab2:
        n = st.slider("Sample size", 100, 2000, 500, 50)
        if st.button("🔬 Generate & Analyse", type="primary"):
            with st.spinner("Generating and analysing…"):
                df_s = pd.concat([make_normal(int(n*0.7)),make_suspicious(int(n*0.2)),make_malicious(int(n*0.1))],ignore_index=True).sample(frac=1,random_state=42)
                base = datetime.now()-timedelta(hours=3)
                df_s["timestamp"] = [(base+timedelta(seconds=i*5)).strftime("%Y-%m-%d %H:%M:%S") for i in range(len(df_s))]
                df_s["src_ip"] = [(random_external_ip() if random.random()<0.2 else f"192.168.{random.randint(0,255)}.{random.randint(1,254)}") for _ in range(len(df_s))]
                df_s["dst_ip"] = [f"10.0.{random.randint(0,10)}.{random.randint(1,50)}" for _ in range(len(df_s))]
                results = classify_batch(df_s, scaler, rf, xgb_model, ann)
                results["src_country"] = results["src_ip"].apply(lambda ip: geo_lookup(str(ip)).get("country",""))
                results["src_flag"]    = results["src_ip"].apply(lambda ip: geo_lookup(str(ip)).get("flag","🌐"))
            st.session_state["csv_results"] = results
            st.success(f"✅ Analysed {len(results):,} packets!")

    # ═══════════════════════════════════════════════════════════════════════
    # TAB 3 — MANUAL ANALYSIS (as-is, just moved inside tab)
    # ═══════════════════════════════════════════════════════════════════════
    with tab3:
        st.markdown("## 🧪 Manual Traffic Analysis")
        st.info("Enter key features only — system auto-generates advanced network behavior")

        if "manual_history" not in st.session_state:
            st.session_state.manual_history = []

        with st.form("manual_form"):
            c1, c2, c3 = st.columns(3)

            with c1:
                duration = st.number_input("Duration", value=10.0)
                src_bytes = st.number_input("Src Bytes", value=500)
                dst_bytes = st.number_input("Dst Bytes", value=1000)

            with c2:
                wrong_fragment = st.number_input("Wrong Fragment", value=0)
                urgent = st.number_input("Urgent", value=0)
                failed_logins = st.number_input("Failed Logins", value=0)

            with c3:
                logged_in = st.selectbox("Logged In", [0,1])
                protocol_type = st.selectbox("Protocol", ["tcp","udp","icmp"])
                packet_size = st.number_input("Packet Size", value=800)

            submit = st.form_submit_button("🚀 Analyse Packet")

        if submit:
            protocol_map = {"tcp":0, "udp":1, "icmp":2}
            protocol_val = protocol_map[protocol_type]

            count = 50 if src_bytes > 1000 else 5
            srv_count = count
            serror_rate = 0.9 if failed_logins > 2 else 0.1
            rerror_rate = 0.8 if wrong_fragment > 1 else 0.05
            same_srv_rate = 0.2 if src_bytes > 1000 else 0.8
            diff_srv_rate = 1 - same_srv_rate
            dst_host_count = 200 if src_bytes > 1000 else 20
            dst_host_srv_count = dst_host_count
            dst_host_same_srv_rate = same_srv_rate
            dst_host_diff_srv_rate = diff_srv_rate
            dst_host_serror_rate = serror_rate
            dst_host_rerror_rate = rerror_rate
            src_port = random.randint(1024, 65535)
            dst_port = 80
            land = 0
            hot = failed_logins
            num_compromised = failed_logins // 2

            input_data = pd.DataFrame([{
                "duration": duration,
                "protocol_type": protocol_val,
                "src_bytes": src_bytes,
                "dst_bytes": dst_bytes,
                "land": land,
                "wrong_fragment": wrong_fragment,
                "urgent": urgent,
                "hot": hot,
                "num_failed_logins": failed_logins,
                "logged_in": logged_in,
                "num_compromised": num_compromised,
                "count": count,
                "srv_count": srv_count,
                "serror_rate": serror_rate,
                "rerror_rate": rerror_rate,
                "same_srv_rate": same_srv_rate,
                "diff_srv_rate": diff_srv_rate,
                "dst_host_count": dst_host_count,
                "dst_host_srv_count": dst_host_srv_count,
                "dst_host_same_srv_rate": dst_host_same_srv_rate,
                "dst_host_diff_srv_rate": dst_host_diff_srv_rate,
                "dst_host_serror_rate": dst_host_serror_rate,
                "dst_host_rerror_rate": dst_host_rerror_rate,
                "src_port": src_port,
                "dst_port": dst_port,
                "packet_size": packet_size
            }])

            input_data = input_data[FEATURE_COLS]

            result = classify_batch(input_data, scaler, rf, xgb_model, ann).iloc[0]
            label = result["predicted_label_name"]
            risk  = result["risk_score"]
            conf  = result["confidence"]

            if label == "Malicious":
                st.error(f"""🚨 MALICIOUS ATTACK\nRisk: {risk:.2f} | Confidence: {conf:.2%}""")
            elif label == "Suspicious":
                st.warning(f"""⚠️ SUSPICIOUS\nRisk: {risk:.2f} | Confidence: {conf:.2%}""")
            else:
                st.success(f"""✅ NORMAL\nRisk: {risk:.2f} | Confidence: {conf:.2%}""")

            st.markdown("### 🧠 Why?")
            reasons = []
            if src_bytes > 1500:      reasons.append("High traffic volume")
            if failed_logins > 2:     reasons.append("Multiple failed logins")
            if wrong_fragment > 1:    reasons.append("Fragmentation anomaly")
            if duration < 2:          reasons.append("Burst traffic pattern")
            for r in reasons:
                st.markdown(f"- {r}")
            if not reasons:
                st.markdown("- No strong anomaly detected")

            st.markdown("### 📊 Feature Risk")

            def bar(val, max_val):
                pct = min(val/max_val, 1)
                return f"<div style='background:#30363d'><div style='width:{pct*100}%;background:#ff4444;padding:4px'></div></div>"

            st.markdown("Src Bytes")
            st.markdown(bar(src_bytes, 3000), unsafe_allow_html=True)
            st.markdown("Failed Logins")
            st.markdown(bar(failed_logins, 5), unsafe_allow_html=True)
            st.markdown("Duration")
            st.markdown(bar(duration, 60), unsafe_allow_html=True)

            st.markdown("""### 🛡 System Action""")
            if label == "Malicious":
                st.error("""✔ Firewall Rule Applied | Traffic Blocked""")
            elif label == "Suspicious":
                st.warning("""✔ Monitoring Enabled""")
            else:
                st.success("""✔ Safe Traffic""")

            st.session_state.manual_history.append({
                "Label": label,
                "Risk": round(risk, 2),
                "Confidence": f"{conf:.2%}"
            })

        if st.session_state.manual_history:
            st.markdown("### 📜 Recent Checks")
            st.dataframe(pd.DataFrame(st.session_state.manual_history[::-1][:10]), use_container_width=True)

    # ═══════════════════════════════════════════════════════════════════════
    # CSV RESULTS (Upload/Sample ke baad dikhta hai — tab ke bahar, same as before)
    # ═══════════════════════════════════════════════════════════════════════
    if "csv_results" in st.session_state:
        results = st.session_state["csv_results"]
        st.divider()
        c1,c2,c3,c4 = st.columns(4)
        c1.metric("Total",f"{len(results):,}")
        c2.metric("✅ Normal",   f"{(results['predicted_label_name']=='Normal').sum():,}")
        c3.metric("⚠️ Suspicious",f"{(results['predicted_label_name']=='Suspicious').sum():,}")
        c4.metric("🚨 Malicious", f"{(results['predicted_label_name']=='Malicious').sum():,}")

        col1,col2 = st.columns(2)
        with col1:
            counts = results["predicted_label_name"].value_counts().reset_index()
            counts.columns=["Label","Count"]
            fig = px.pie(counts,names="Label",values="Count",color="Label",color_discrete_map=COLOR_MAP,title="Traffic Distribution")
            fig.update_layout(paper_bgcolor="#161b22",font_color="#c9d1d9")
            st.plotly_chart(fig, use_container_width=True)
        with col2:
            fig2 = px.histogram(results,x="risk_score",color="predicted_label_name",color_discrete_map=COLOR_MAP,nbins=30,title="Risk Score Distribution")
            fig2.update_layout(paper_bgcolor="#161b22",plot_bgcolor="#0d1117",font_color="#c9d1d9")
            st.plotly_chart(fig2, use_container_width=True)

        if "src_country" in results.columns:
            top = results[results["predicted_label_name"]!="Normal"]["src_country"].value_counts().head(10).reset_index()
            top.columns=["Country","Threats"]
            st.subheader("🌍 Top Threat Origins")
            st.dataframe(top, use_container_width=True)

        st.divider()
        if st.button("🔍 Attack Distribution", type="primary", use_container_width=True):
            st.session_state["show_attack_dist"] = True

        if st.session_state.get("show_attack_dist", False):
            st.markdown("### 🎯 Attack Type Distribution")
            if "attack_type" in results.columns:
                df_attacks = results[
                    (results["attack_type"].notna()) &
                    (results["attack_type"].str.strip() != "") &
                    (results["attack_type"].str.strip().str.lower() != "normal")
                ].copy()
                if len(df_attacks) == 0:
                    st.warning("⚠️ Koi attack record nahi mila. Sabhi entries Normal hain.")
                else:
                    attack_counts = df_attacks["attack_type"].value_counts().reset_index()
                    attack_counts.columns = ["Attack Type", "Count"]
                    ATTACK_COLORS = {"DoS":"#ff4444","Probe":"#ffa500","R2L":"#a855f7","U2R":"#ec4899"}
                    col_a, col_b = st.columns(2)
                    with col_a:
                        fig_atk_pie = px.pie(attack_counts,names="Attack Type",values="Count",color="Attack Type",color_discrete_map=ATTACK_COLORS,title="🥧 Attack Type Share",hole=0.35)
                        fig_atk_pie.update_traces(textposition="inside",textinfo="percent+label")
                        fig_atk_pie.update_layout(paper_bgcolor="#161b22",font_color="#c9d1d9",legend=dict(bgcolor="#0d1117"))
                        st.plotly_chart(fig_atk_pie, use_container_width=True)
                    with col_b:
                        fig_atk_bar = px.bar(attack_counts,x="Attack Type",y="Count",color="Attack Type",color_discrete_map=ATTACK_COLORS,title="📊 Attack Type Count",text="Count")
                        fig_atk_bar.update_traces(textposition="outside")
                        fig_atk_bar.update_layout(paper_bgcolor="#161b22",plot_bgcolor="#0d1117",font_color="#c9d1d9",showlegend=False,xaxis=dict(gridcolor="#21262d"),yaxis=dict(gridcolor="#21262d"))
                        st.plotly_chart(fig_atk_bar, use_container_width=True)
                    st.markdown("#### 📋 Attack Breakdown")
                    attack_counts["Percentage"] = (attack_counts["Count"] / attack_counts["Count"].sum() * 100).round(2).astype(str) + "%"
                    st.dataframe(attack_counts, use_container_width=True)
            else:
                st.error("❌ 'attack_type' column results mein nahi mili.")

        show_cols = [c for c in ["timestamp","src_ip","src_country","dst_ip","predicted_label_name","spam_flag","confidence","risk_score","attack_type"] if c in results.columns]
        flt = st.multiselect("Filter", ["Normal","Suspicious","Malicious"], default=["Suspicious","Malicious"])
        st.dataframe(results[results["predicted_label_name"].isin(flt)][show_cols] if flt else results[show_cols], use_container_width=True)
        st.download_button("📥 Download CSV", results.to_csv(index=False).encode(), "ids_results.csv", "text/csv", type="primary")


# ══════════════ THREAT MAP (FINAL ADVANCED) ═══════════════════════════════════
elif page == "🗺️ Threat Map":

    import time

    st.markdown("## 🌐 Real-Time Cyber Threat Intelligence Map")

    geo_records = []

    # ── LIVE MONITOR DATA ───────────────────────────────────────
    for e in st.session_state.get("live_log", []):

        if e["label"] in ("Suspicious", "Malicious"):

            g = e.get("src_geo", {})

            if g and g.get("lat", 0) != 0.0:

                geo_records.append({
                    "ip": e["src_ip"],
                    "country": g.get("country", "Unknown"),
                    "lat": g["lat"],
                    "lon": g["lon"],
                    "label": e["label"],
                    "risk": float(e["risk_score"])
                })

    # ── CSV ANALYSIS DATA ───────────────────────────────────────
    csv_res = st.session_state.get("csv_results")

    if csv_res is not None and "src_ip" in csv_res.columns:

        attack_rows = csv_res[
            csv_res["predicted_label_name"] != "Normal"
        ]

        for _, row in attack_rows.iterrows():

            g = geo_lookup(str(row["src_ip"]))

            if g and g.get("lat", 0) != 0.0:

                geo_records.append({
                    "ip": row["src_ip"],
                    "country": g.get("country", "Unknown"),
                    "lat": g["lat"],
                    "lon": g["lon"],
                    "label": row["predicted_label_name"],
                    "risk": float(row["risk_score"])
                })

    # ── NO DATA ─────────────────────────────────────────────────
    if len(geo_records) == 0:

        st.warning("⚠️ No threat data available yet.")
        st.info("Start Live Monitor or run CSV Analysis first.")

    else:

        df_geo = pd.DataFrame(geo_records)

        # ── 🚨 HIGH RISK ALERT ──────────────────────────────────
        high_risk = df_geo[df_geo["risk"] > 85]

        if not high_risk.empty:

            st.error(
                f"🚨 {len(high_risk)} High-Risk Attack(s) Detected!"
            )

        # ── LAYOUT ─────────────────────────────────────────────
        col1, col2 = st.columns([4, 1])

        # ── 🌍 MAP SECTION ─────────────────────────────────────
        with col1:

            DEST_LAT = 20.59
            DEST_LON = 78.96

            fig = px.scatter_geo(
                df_geo,
                lat="lat",
                lon="lon",
                color="label",
                size="risk",
                hover_name="country",
                hover_data={
                    "ip": True,
                    "risk": True,
                    "lat": False,
                    "lon": False
                },
                projection="natural earth"
            )

            # 🔥 ATTACK FLOW LINES
            for _, row in df_geo.iterrows():

                fig.add_trace(
                    px.line_geo(
                        lat=[row["lat"], DEST_LAT],
                        lon=[row["lon"], DEST_LON]
                    ).data[0]
                )

            # 🌌 CYBER UI
            fig.update_layout(

                title="🌐 Global Threat Origins & Attack Flow",

                paper_bgcolor="#0b0f19",

                geo=dict(
                    bgcolor="#0b0f19",
                    showland=True,
                    landcolor="#161b22",
                    showocean=True,
                    oceancolor="#0b0f19",
                    showcountries=True,
                    countrycolor="#30363d",
                    showframe=False
                ),

                font_color="#00ffee",

                height=550
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

        # ── 📊 TOP COUNTRIES PANEL ─────────────────────────────
        with col2:

            st.markdown("## 📊 Top Attack Countries")

            top_countries = (
                df_geo["country"]
                .value_counts()
                .head(5)
            )

            for i, (country, count) in enumerate(
                top_countries.items(),
                start=1
            ):

                st.markdown(
                    f"**{i}. {country}** — {count} attacks"
                )

        # ── ⏱ AUTO REFRESH ────────────────────────────────────
        time.sleep(5)

        try:
            st.rerun()
        except:
            pass

# ══════════════ MODEL INFO ════════════════════════════════════════════════════
elif page == "🧠 Model Info":
    st.markdown("## 🧠 Model Architecture")
    st.markdown("""
    | Model | Type | Details |
    |-------|------|---------|
    | 🌲 Random Forest | Ensemble Trees | 100 trees, depth 15, class-weighted |
    | ⚡ XGBoost | Gradient Boost | 150 trees, lr=0.08, depth 6 |
    | 🧠 ANN (MLP) | Deep Learning | 256→128→64→32, ReLU, early stopping |
    | 🗳️ Ensemble | Soft Voting | Average of all 3 probabilities |

    ### 🏷️ Labels
    | Label | Risk | Attacks |
    |-------|------|---------|
    | ✅ Normal | 0–39 | Legitimate |
    | ⚠️ Suspicious | 40–74 | Probe, R2L |
    | 🚨 Malicious | 75–100 | DoS, U2R |
    """)
    st.subheader("🔬 Single Packet Classifier")
    with st.form("classify"):
        c1,c2,c3 = st.columns(3)
        with c1:
            duration=st.number_input("Duration",0.0,10000.0,50.0)
            src_bytes=st.number_input("Src Bytes",0.0,1e6,5000.0)
            dst_bytes=st.number_input("Dst Bytes",0.0,1e6,5000.0)
        with c2:
            serr=st.slider("SError Rate",0.0,1.0,0.05)
            rerr=st.slider("RError Rate",0.0,1.0,0.05)
            same=st.slider("Same Srv Rate",0.0,1.0,0.90)
        with c3:
            fl=st.number_input("Failed Logins",0,20,0)
            li=st.selectbox("Logged In",[1,0])
            proto=st.selectbox("Protocol",[0,1,2],format_func=lambda x:["TCP","UDP","ICMP"][x])
        submitted=st.form_submit_button("🚀 Classify", type="primary")
    if submitted:
        d={c:0 for c in FEATURE_COLS}
        d.update({"duration":duration,"src_bytes":src_bytes,"dst_bytes":dst_bytes,
                  "serror_rate":serr,"rerror_rate":rerr,"same_srv_rate":same,
                  "num_failed_logins":fl,"logged_in":li,"protocol_type":proto})
        res=classify_single(d,scaler,rf,xgb_model,ann)
        lbl=res["label"]
        if lbl=="Normal": st.success(f"✅ {lbl} | {res[\'spam\']} | Risk: {res[\'risk_score\']} | Conf: {res[\'confidence\']:.1%}")
        elif lbl=="Suspicious": st.warning(f"⚠️ {lbl} | {res[\'spam\']} | Risk: {res[\'risk_score\']} | Conf: {res[\'confidence\']:.1%}")
        else: st.error(f"🚨 {lbl} | {res[\'spam\']} | Risk: {res[\'risk_score\']} | Conf: {res[\'confidence\']:.1%}")
        st.write(f"RF: {res[\'rf_vote\']}  |  XGB: {res[\'xgb_vote\']}  |  ANN: {res[\'ann_vote\']}")
        fig=go.Figure(go.Bar(x=["Normal","Suspicious","Malicious"],y=res["proba"],marker_color=["#00cc88","#ffa500","#ff4444"]))
        fig.update_layout(paper_bgcolor="#161b22",plot_bgcolor="#0d1117",font_color="#c9d1d9",height=220,margin=dict(t=10,b=20,l=0,r=0))
        st.plotly_chart(fig, use_container_width=True)
'''

with open('ids_app.py', 'w') as f:
    f.write(app_code)
print('✅ ids_app.py written to disk')

✅ ids_app.py written to disk


In [4]:
# ── Cell 4: Launch Streamlit + ngrok tunnel ───────────────────────────────────
import subprocess, threading, time
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'ids_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for Streamlit to start
time.sleep(4)

# Open ngrok tunnel
public_url = ngrok.connect(8501)

print("=" * 55)
print("🛡️ NETWORK IDS IS LIVE!")
print("=" * 55)

print(f"\n🌐 Public URL: https://rust-kinship-flying.ngrok-free.dev\n")

print("Click the link above to open your IDS dashboard!")
print("\nKeep this cell running to stay live.")

🛡️ NETWORK IDS IS LIVE!

🌐 Public URL: https://rust-kinship-flying.ngrok-free.dev

Click the link above to open your IDS dashboard!

Keep this cell running to stay live.


In [5]:
# ── Cell 5 (Optional): Stop the app ──────────────────────────────────────────
# Run this cell when you want to shut everything down
# ngrok.kill()
# proc.terminate()
# print('✅ App stopped')